# Modelo analítico: riesgo y segmentación de clientes

**Preguntas que responde el modelo**

1. ¿Qué tan riesgoso es el portafolio de cada cliente en su última fecha disponible?
2. ¿Ese riesgo está alineado con el perfil de riesgo que tiene registrado?
3. ¿Qué grupos (segmentos) de clientes se parecen entre sí?

**Cómo se construye**

| Paso | Qué se hace | Dónde |
|---|---|---|
| 1 | Se asigna un nivel de riesgo (bajo, medio, alto) a cada activo con información de mercado | `sql/asset-risk.sql` |
| 2 | Se calcula el promedio ponderado del riesgo de cada portafolio y se compara con el perfil | `sql/client-risk.sql` |
| 3 | Se agrupan los clientes con K-means | este notebook |

**Reglas acordadas**

- Nivel por volatilidad anual con las clases de riesgo de mercado de PRIIPs: bajo < 5 % (clases 1 y 2), medio de 5 % a menos de 20 % (clases 3 y 4), alto 20 % o más (clases 5 a 7).
- Acciones colombianas: volatilidad calculada con los datos entregados. FICs y CDT: perfil publicado por la entidad. Activos internacionales: volatilidad publicada del fondo o, si no hay, la de su clase de activo.
- Puntajes: bajo = 1, medio = 2, alto = 3. El riesgo del portafolio es el promedio ponderado por saldo y se lleva al nivel más cercano.
- Perfil conservador = bajo, moderado = medio, agresivo = alto. Los clientes `SIN DEFINIR` quedan como "Sin perfil para comparar".
- COP y USD se analizan por separado (no se convierte con la TRM).
- Los 11 clientes con ID en notación científica se incluyen y se marcan como ID no confiable.

In [ ]:
import os
from decimal import Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# El notebook vive en notebooks/, la raíz del proyecto está un nivel arriba
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(BASE_DIR / ".env")

connection = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    dbname=os.getenv("PGDATABASE"),
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Colores de las gráficas
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
LEVEL_COLORS = {"bajo": "#86b6ef", "medio": "#2a78d6", "alto": "#104281"}
GRAY = "#898781"
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.6,
    "figure.dpi": 110,
})


def run_sql_file(file_name):
    # Ejecuta un archivo .sql completo dentro de una transacción
    sql_text = (BASE_DIR / "sql" / file_name).read_text(encoding="utf-8")
    with connection.cursor() as cursor:
        cursor.execute(sql_text)
    connection.commit()
    print(f"Script '{file_name}' ejecutado correctamente.")


def query(sql_text):
    # Ejecuta una consulta y devuelve un DataFrame con los números como float
    with connection.cursor() as cursor:
        cursor.execute(sql_text)
        columns = [column.name for column in cursor.description]
        data = pd.DataFrame(cursor.fetchall(), columns=columns)
    for column in data.columns:
        if data[column].map(lambda value: isinstance(value, Decimal)).any():
            data[column] = data[column].astype(float)
    return data

## Paso 1. Nivel de riesgo de cada activo

`sql/asset-risk.sql` crea la tabla `activos_riesgo` con el nivel de cada activo, el método usado, el dato de referencia y la URL de la fuente. `sql/client-risk.sql` crea la vista `riesgo_clientes`. Ambos se ejecutan aquí para que el notebook sea reproducible (requiere haber corrido antes `sql/clear.sql`).

In [ ]:
run_sql_file("asset-risk.sql")
run_sql_file("client-risk.sql")

assets = query("SELECT * FROM activos_riesgo ORDER BY moneda, tipo_activo, nombre")
print(f"Activos clasificados: {len(assets)}")
assets.pivot_table(index=["moneda", "metodo"], columns="nivel_riesgo", values="id",
                   aggfunc="count", fill_value=0)[["bajo", "medio", "alto"]]

### Validación de la volatilidad de las acciones colombianas

La volatilidad se calcula con el cambio diario del ABA de cada cliente en cada acción. Un salto grande podría ser una compra o venta y no un movimiento de precio. Si el salto ocurre el mismo día y con el mismo porcentaje en todos los clientes que tienen la acción, es un cambio de precio real y debe quedar en el cálculo.

In [ ]:
jumps = query('''
    WITH serie AS (
        SELECT cod_activo, id_sistema_cliente, fecha_ingesta, aba,
               LAG(aba) OVER (PARTITION BY id_sistema_cliente, cod_activo ORDER BY fecha_ingesta) AS aba_anterior
        FROM macroactivos_limpio
        WHERE macroactivo = 'Renta Variable' AND cod_activo IS NOT NULL
    )
    SELECT a.activo, s.fecha_ingesta, COUNT(*) AS clientes,
           STRING_AGG(ROUND((s.aba / s.aba_anterior - 1) * 100, 1)::TEXT, ' | ') AS cambio_pct_por_cliente
    FROM serie s
    JOIN catalogo_activos a ON a.cod_activo = s.cod_activo
    WHERE s.aba_anterior > 0 AND ABS(s.aba / s.aba_anterior - 1) >= 0.10
    GROUP BY a.activo, s.fecha_ingesta
    ORDER BY a.activo, s.fecha_ingesta
''')
print("Cambios diarios de 10 % o más en acciones colombianas:")
jumps

In [ ]:
print("Los saltos son iguales en todos los clientes de la misma acción: son movimientos de precio y no se eliminan.\n")
stocks = assets[assets["tipo_activo"] == "Acción colombiana"]
stocks[["cod_activo", "nombre", "metodo", "volatilidad_anual", "nivel_riesgo", "referencia"]]

In [ ]:
# Tabla completa de referencia (la URL de cada fuente está en la columna fuente)
assets[["moneda", "nombre", "tipo_activo", "metodo", "volatilidad_anual", "nivel_riesgo", "referencia", "fuente"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
summary = assets.groupby(["moneda", "nivel_riesgo"]).size().unstack(fill_value=0)[["bajo", "medio", "alto"]]
left = np.zeros(len(summary))
for level in ["bajo", "medio", "alto"]:
    values = summary[level].values
    ax.barh(summary.index, values, left=left, color=LEVEL_COLORS[level], label=level,
            height=0.5, edgecolor="#fcfcfb", linewidth=2)
    for position, (start, value) in enumerate(zip(left, values)):
        if value > 0:
            ax.text(start + value / 2, position, str(value), ha="center", va="center",
                    color="#0b0b0b" if level == "bajo" else "#ffffff", fontsize=9)
    left += values
ax.set_title("Activos por moneda y nivel de riesgo", loc="left", fontsize=11)
ax.set_xlabel("Cantidad de activos")
ax.set_axisbelow(True)
ax.legend(title="Nivel", frameon=False, loc="center left", bbox_to_anchor=(1, 0.5))
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## Paso 2. Riesgo del portafolio y alineación con el perfil

La vista `riesgo_clientes` toma la última fecha de cada cliente y calcula, por separado para COP y USD:

- `puntaje = Σ(saldo × puntaje del activo) / Σ(saldo de los activos clasificados)`
- el nivel más cercano: bajo si el puntaje es menor que 1,5; medio si es menor que 2,5; alto en otro caso
- la comparación con el perfil declarado

Los activos sin código de catálogo no tienen nivel y quedan por fuera del promedio; su peso se muestra en `pct_sin_clasificar_cop`.

In [ ]:
clients = query("SELECT * FROM riesgo_clientes ORDER BY aba_cop DESC")
print(f"Clientes: {len(clients)}  |  con portafolio USD: {clients['valor_usd'].notna().sum()}  |  ID no confiable: {(~clients['id_confiable']).sum()}")
clients[["id_sistema_cliente", "id_confiable", "banca", "perfil_riesgo", "aba_cop", "pct_renta_variable",
         "pct_renta_fija", "pct_fics", "puntaje_cop", "nivel_cop", "pct_sin_clasificar_cop", "alineacion_cop",
         "valor_usd", "puntaje_usd", "nivel_usd", "alineacion_usd"]]

In [ ]:
order = ["Alineado", "Más riesgo que su perfil", "Menos riesgo que su perfil",
         "Sin perfil para comparar", "Perfil no único", "Sin portafolio USD"]
alignment = pd.DataFrame({
    "COP": clients["alineacion_cop"].value_counts(),
    "USD": clients["alineacion_usd"].value_counts(),
}).reindex(order).fillna(0).astype(int)
print("Clientes por resultado de la comparación con el perfil:")
alignment

In [ ]:
compared = clients[clients["alineacion_cop"].isin(order[:3])]
counts = compared["alineacion_cop"].value_counts().reindex(order[:3]).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(8, 2.8))
bars = ax.barh(counts.index[::-1], counts.values[::-1], color=COLORS[:3][::-1], height=0.5)
for bar, value in zip(bars, counts.values[::-1]):
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2, str(value), va="center", fontsize=9)
ax.set_title(f"Portafolio COP frente al perfil declarado ({len(compared)} clientes con perfil definido)", loc="left", fontsize=11)
ax.set_xlabel("Clientes")
ax.set_axisbelow(True)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

In [ ]:
# Cruce entre el perfil declarado y el nivel real del portafolio COP
pd.crosstab(clients["perfil_riesgo"], clients["nivel_cop"]).reindex(columns=["bajo", "medio", "alto"], fill_value=0)

## Paso 3. Segmentación con K-means

**Variables por cliente** (todas salen de la vista `riesgo_clientes`):

| Variable | Por qué |
|---|---|
| `log_aba_cop` | tamaño del portafolio local (en logaritmo porque los saldos van de miles a miles de millones) |
| `log_valor_usd` | tamaño del portafolio internacional (0 si no tiene) |
| `pct_renta_variable`, `pct_renta_fija`, `pct_fics` | composición del portafolio COP |
| `puntaje_cop`, `puntaje_usd` | riesgo ponderado de cada portafolio (0 en USD si no tiene) |
| `activos` | número de activos que tiene |

Las variables se estandarizan para que ninguna pese más solo por su escala. El número de grupos se elige con el coeficiente de silueta (se prueba de 2 a 8 y se toma el más alto).

In [ ]:
features = pd.DataFrame({
    "log_aba_cop": np.log10(clients["aba_cop"].clip(lower=1)),
    "log_valor_usd": np.log10(clients["valor_usd"].fillna(0) + 1),
    "pct_renta_variable": clients["pct_renta_variable"],
    "pct_renta_fija": clients["pct_renta_fija"],
    "pct_fics": clients["pct_fics"],
    "puntaje_cop": clients["puntaje_cop"],
    "puntaje_usd": clients["puntaje_usd"].fillna(0),
    "activos": clients["activos_cop"] + clients["activos_usd"],
})
scaled = StandardScaler().fit_transform(features)

scores = {}
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=20, random_state=42)
    scores[k] = silhouette_score(scaled, model.fit_predict(scaled))

best_k = max(scores, key=scores.get)
print("Coeficiente de silueta por número de grupos:")
for k, value in scores.items():
    print(f"  k = {k}: {value:.3f}{'  <- mejor' if k == best_k else ''}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(list(scores.keys()), list(scores.values()), color=COLORS[0], linewidth=2, marker="o", markersize=7)
ax.scatter([best_k], [scores[best_k]], s=90, color=COLORS[0], zorder=3, edgecolor="#fcfcfb", linewidth=2)
ax.annotate(f"k = {best_k}", (best_k, scores[best_k]), textcoords="offset points", xytext=(8, 6), fontsize=9)
ax.set_title("Coeficiente de silueta según el número de grupos", loc="left", fontsize=11)
ax.set_xlabel("Número de grupos (k)")
ax.set_ylabel("Silueta")
ax.set_xticks(list(scores.keys()))
plt.tight_layout()
plt.show()

In [ ]:
model = KMeans(n_clusters=best_k, n_init=20, random_state=42)
clients["segmento"] = model.fit_predict(scaled) + 1
clients["tiene_usd"] = clients["valor_usd"].notna()

profile = clients.groupby("segmento").agg(
    clientes=("id_sistema_cliente", "count"),
    aba_cop_mediana=("aba_cop", "median"),
    pct_renta_variable=("pct_renta_variable", "mean"),
    pct_renta_fija=("pct_renta_fija", "mean"),
    pct_fics=("pct_fics", "mean"),
    puntaje_cop=("puntaje_cop", "mean"),
    con_usd=("tiene_usd", "mean"),
    puntaje_usd=("puntaje_usd", "mean"),
).round(2)


def segment_name(row):
    # Nombre automático: tipo de activo dominante en COP y si tiene portafolio USD
    labels = {"pct_renta_variable": "Renta variable", "pct_renta_fija": "Renta fija", "pct_fics": "FICs"}
    main = row[list(labels)].idxmax()
    if row["con_usd"] == 1:
        usd = "con USD"
    elif row["con_usd"] == 0:
        usd = "solo COP"
    else:
        usd = "parte con USD"
    return f"{labels[main]} {usd}"


profile["nombre_segmento"] = profile.apply(segment_name, axis=1)

# Si dos segmentos quedan con el mismo nombre se diferencian por el tamaño del ABA
repeated = profile["nombre_segmento"].duplicated(keep=False)
for name in profile.loc[repeated, "nombre_segmento"].unique():
    group = profile[profile["nombre_segmento"] == name].sort_values("aba_cop_mediana")
    for position, segment in enumerate(group.index):
        size = "ABA menor" if position == 0 else "ABA mayor"
        profile.loc[segment, "nombre_segmento"] = f"{name} ({size})"

clients["nombre_segmento"] = clients["segmento"].map(profile["nombre_segmento"])
profile

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
names = [f"{segment}. {name}" for segment, name in profile["nombre_segmento"].items()][::-1]
values = profile["clientes"].values[::-1]
bars = ax.barh(names, values, color=COLORS[0], height=0.55)
for bar, (segment, row) in zip(bars, list(profile.iterrows())[::-1]):
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
            f"{int(row['clientes'])} clientes · riesgo COP {row['puntaje_cop']:.2f}", va="center", fontsize=8.5)
ax.set_title("Clientes por segmento y riesgo COP promedio (1 bajo, 3 alto)", loc="left", fontsize=11)
ax.set_xlabel("Clientes")
ax.set_xlim(0, max(values) + 4)
ax.set_axisbelow(True)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

In [ ]:
# Guardar el segmento de cada cliente para usarlo en la aplicación Django
with connection.cursor() as cursor:
    cursor.execute('''
        DROP TABLE IF EXISTS segmentos_clientes;
        CREATE TABLE segmentos_clientes (
            id_sistema_cliente VARCHAR(30) PRIMARY KEY,
            segmento           SMALLINT    NOT NULL,
            nombre_segmento    VARCHAR(80) NOT NULL
        );
    ''')
    cursor.executemany(
        "INSERT INTO segmentos_clientes (id_sistema_cliente, segmento, nombre_segmento) VALUES (%s, %s, %s)",
        clients[["id_sistema_cliente", "segmento", "nombre_segmento"]].values.tolist(),
    )
connection.commit()
print(f"Tabla segmentos_clientes guardada con {len(clients)} clientes.")

## Hallazgos

Los números se calculan con los datos para que el texto siempre coincida con la tabla.

In [ ]:
with_profile = clients[clients["alineacion_cop"].isin(order[:3])]
more_risk = with_profile[with_profile["alineacion_cop"] == "Más riesgo que su perfil"]
less_risk = with_profile[with_profile["alineacion_cop"] == "Menos riesgo que su perfil"]
no_profile = clients[clients["alineacion_cop"] == "Sin perfil para comparar"]
usd_clients = clients[clients["valor_usd"].notna()]
total_aba = clients["aba_cop"].sum()

print(f"1. De {len(clients)} clientes, {len(with_profile)} tienen un perfil que se puede comparar.")
print(f"   - Alineados: {(with_profile['alineacion_cop'] == 'Alineado').sum()}")
print(f"   - Con más riesgo que su perfil: {len(more_risk)} (perfiles: {', '.join(more_risk['perfil_riesgo'])})")
print(f"   - Con menos riesgo que su perfil: {len(less_risk)}, que suman "
      f"{less_risk['aba_cop'].sum() / total_aba * 100:.1f} % del ABA COP total")
print(f"2. {len(no_profile)} clientes no tienen perfil definido; "
      f"{(no_profile['nivel_cop'] == 'alto').sum()} de ellos tienen un portafolio COP de riesgo alto.")
print(f"3. {len(usd_clients)} clientes tienen portafolio internacional; niveles USD: "
      f"{usd_clients['nivel_usd'].value_counts().to_dict()}")
print(f"4. Activos sin clasificar (sin código de catálogo): "
      f"{(clients['pct_sin_clasificar_cop'] > 0).sum()} clientes los tienen.")
print(f"5. Segmentos encontrados (k = {best_k}):")
for segment, row in profile.iterrows():
    print(f"   {segment}. {row['nombre_segmento']}: {int(row['clientes'])} clientes, "
          f"puntaje COP promedio {row['puntaje_cop']:.2f}")

## Limitaciones

- Las volatilidades publicadas de fondos y ETF son las disponibles en 2025 y 2026, no las de mayo de 2024 (fecha de los datos).
- Para bonos individuales, acciones de EE.UU. y notas estructuradas se usa la volatilidad de su clase de activo. Un bono de un solo emisor puede ser más riesgoso que un índice diversificado.
- Algunas volatilidades vienen de otra clase del mismo fondo (por ejemplo, la clase cubierta en EUR); la referencia lo indica.
- CEMARGOS solo tiene 4 retornos diarios; se usa el promedio de las demás acciones colombianas del archivo.
- El FIC sin código y las acciones con código 1022 o sin código no se pueden identificar, así que quedan sin clasificar.
- Los 11 IDs en notación científica pueden mezclar clientes distintos (por ejemplo `1.00114E+12` tiene dos perfiles); se incluyen marcados como no confiables.
- Con 30 clientes los segmentos son descriptivos; no se deben usar para predecir sin más datos.

## Fuentes principales

Cada activo tiene su URL en la columna `fuente` de `activos_riesgo`. Las referencias generales son:

- Escala de riesgo por volatilidad: Reglamento Delegado (UE) 2017/653 (PRIIPs), anexo II, clases MRM: https://handbook.fca.org.uk/techstandards/PRIIPs/2017/reg_del_2017_653_oj/annex02.html
- Perfiles de FICs y CDT: Fiduciaria Bancolombia, Bancolombia, Sura Investments y Fogafín.
- Fondos y ETF: BlackRock / iShares, Franklin Templeton, Ninety One, Schroders, J.P. Morgan, PIMCO (vía easybank, finanzen.net, onvista) y Seeking Alpha.
- Clases de activo: VIX EQ (J.P. Morgan Asset Management), iShares CEMB y EMB, Vanguard VGSH, Nasdaq-100 (BNP Paribas).

In [ ]:
connection.close()
print("Conexión cerrada.")